<a href="https://colab.research.google.com/github/trongtin142/cofog-panel/blob/main/cofog.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# COFOG-Panel in Colab

Install the package, upload the IMF extract, choose an indicator in the form and run the pipeline.

1. **Install** the tagged release.
2. **Data**: upload `gfs_raw_data.xlsx` to `data/` (the country lookup is in the repository).
3. **Form**: COFOG function, transformation, method and output column names.
4. **Run** all five stages, or the single stages in section 5.

The IMF data are subject to the IMF terms of use (see `DATA_LICENSE.md` in the repository).

In [ ]:
# @title 1. Install
# Tagged release, for reproducible results (use the version cited in your work)
RELEASE = "v1.0.5"
!git clone --depth 1 --branch {RELEASE} https://github.com/trongtin142/cofog-panel.git
%cd cofog-panel
!pip install -q -e .
!cofog-panel --version

In [ ]:
# @title 2. Data
# @markdown Download IMF.STA:GFS_COFOG (https://data.imf.org/en/datasets/IMF.STA:GFS_COFOG, DOWNLOAD, CSV)
# @markdown and upload it as `data/gfs_raw_data.csv` with the Files panel, then run this cell.
from pathlib import Path

for path in (Path("data/gfs_raw_data.csv"), Path("data/gfs_raw_data.xlsx")):
    print(f"{path}: {'found' if path.exists() else 'MISSING'}")

In [ ]:
# @title 3. Form
# @markdown * **COFOG**: division (05 Environmental protection, 07 Health, ...) or group code (07.3).
# @markdown * **Transformation**: shares of total outlays are never summed across subsectors.
# @markdown * **Method**: 'General government' applies the three-tier rule; the other options give one sector.
# @markdown * **Suffix**: names the output columns, e.g. 05 gives DATA_05, SOURCE_TIER_05,
# @markdown   SOURCE_SECTORS_05 and SOURCE_COMPLETE_05. Use a new suffix for each indicator.
import ipywidgets as widgets
from IPython.display import display

DIVISIONS = {
    "01": "General public services", "02": "Defence", "03": "Public order and safety",
    "04": "Economic affairs", "05": "Environmental protection", "06": "Housing and community amenities",
    "07": "Health", "08": "Recreation, culture and religion", "09": "Education", "10": "Social protection",
}
METHODS = ["General government", "Central government including social security",
           "Central government excluding social security", "Budgetary central government",
           "Extrabudgetary central government", "Social security funds", "State Government", "Local Government"]

cofog_box = widgets.Combobox(value="05", options=list(DIVISIONS), description="COFOG:", ensure_option=False)
transformation_box = widgets.Dropdown(options=["Percent of GDP", "Domestic currency", "Percent of total outlays"],
                                      description="Transformation:")
method_box = widgets.Dropdown(options=METHODS, description="Method:")
suffix_box = widgets.Text(value="05", description="Suffix:")
display(cofog_box, transformation_box, method_box, suffix_box)

In [ ]:
# @title Selected settings
COFOG = cofog_box.value.strip()
TRANSFORMATION = transformation_box.value
METHOD = method_box.value
DATA_COL, TIER_COL = f"DATA_{suffix_box.value.strip()}", f"SOURCE_TIER_{suffix_box.value.strip()}"
print(COFOG, DIVISIONS.get(COFOG, ""), "|", TRANSFORMATION, "|", METHOD, "|", DATA_COL, TIER_COL)

In [ ]:
# @title 4. Run all five stages
# @markdown An existing master is kept and the new indicator is added to it.
!cofog-panel run --cofog "{COFOG}" --filter-type "{TRANSFORMATION}" --sector "{METHOD}" --data-col "{DATA_COL}" --tier-col "{TIER_COL}"

## 5. Single stages

In [ ]:
# @title 5a. Stages 1 and 2: checks
!cofog-panel check-format
!cofog-panel check-country-format

In [ ]:
# @title 5b. Stage 3: new master panel (replaces an existing master)
!cofog-panel seed-master --output-dir output

In [ ]:
# @title 5c. Stage 4: one file per country
!cofog-panel split --cofog "{COFOG}" --filter-type "{TRANSFORMATION}" --output-dir intermediate_splits

In [ ]:
# @title 5d. Stage 5: selection rule and provenance
!cofog-panel aggregate --master-file output/COFOG_MASTER_SCHEMA.xlsx --folder-path intermediate_splits --data-col "{DATA_COL}" --tier-col "{TIER_COL}" --sector "{METHOD}"

In [ ]:
# @title 6. Preview the master
import pandas as pd

master = pd.read_excel("output/COFOG_MASTER_SCHEMA.xlsx", sheet_name="MasterData")
print(master[TIER_COL].value_counts())
master.dropna(subset=[DATA_COL]).head(10)